In [117]:
import pandas as pd

In [118]:
df = pd.read_csv('../data/raw/voyage_p3.csv')

In [ ]:
df_dummy = df[df['job_id'].isin(range(1, 26))]
df_dummy.to_csv('../data/processed/voyage_sim.csv', index=False)

# Cek pasang surut

In [120]:
import pandas as pd
import numpy as np

# =========================
# LOAD DATA
# =========================
# voyage = pd.read_csv("../data/raw/voyage_p3.csv")
voyage = pd.read_csv("../data/processed/voyage_dummy.csv")
tidal_lookup = pd.read_csv("../tidal_lookup_debug.csv")
tidal_rules = pd.read_csv("../data/raw/tidal_rules.csv")

# =========================
# NORMALISASI
# =========================
tidal_lookup['is_allowed'] = tidal_lookup['is_allowed'].astype(bool)

voyage['arrival_time'] = voyage['arrival_time'].astype(int)
voyage['due_date'] = voyage['due_date'].astype(int)
voyage['buffer_time'] = voyage['buffer_time'].astype(int)
tidal_lookup['t'] = tidal_lookup['t'].astype(int)

# =========================
# HANYA SHIP + PORT YANG ADA RULE
# =========================
valid_rule_pairs = set(
    zip(tidal_rules['port_name'], tidal_rules['ship_name'])
)

# =========================
# BUILD LOOKUP
# =========================
lookup = {
    (row.port_name, row.ship_name, row.t): row.is_allowed
    for row in tidal_lookup.itertuples()
}

def check_allowed(port, ship, t):
    return lookup.get((port, ship, t), False)


# =========================
# VERIFICATION
# =========================
results = []

for row in voyage.itertuples():

    port = row.port_name
    ship = row.ship_name

    # SKIP jika tidak ada rule
    if (port, ship) not in valid_rule_pairs:
        continue

    arr = row.arrival_time
    due = row.due_date
    buffer = row.buffer_time

    valid = True
    failed_t = []

    if buffer > 0:
        before_range = range(arr - buffer, arr)
        after_range = range(due + 1, due + buffer + 1)

        for t in list(before_range) + list(after_range):
            if not check_allowed(port, ship, t):
                valid = False
                failed_t.append(t)

    else:
        for t in range(arr, due + 1):
            if not check_allowed(port, ship, t):
                valid = False
                failed_t.append(t)

    results.append({
        "job_id": row.job_id,
        "ship_name": ship,
        "port": port,
        "op_seq": row.op_seq,
        "valid": valid,
        "failed_t": failed_t
    })

# =========================
# RESULT
# =========================
result_df = pd.DataFrame(results)

print("SUMMARY")
print(result_df['valid'].value_counts())

print("\nFAILED ROWS")
display(result_df[result_df['valid'] == False])

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/voyage_dummy.csv'

In [ ]:
# =========================
# LOAD TIDAL DATA
# =========================
import pandas as pd
import numpy as np
import re

tidal_data = pd.read_csv("/home/re1jie/TA-code/JSSP-CAOA-SSR/data/tidal_data.csv")

def fix_24_hour(ts):
    if " 24:" in ts:
        date_part, time_part = ts.split(" ")
        # ubah ke 00 dan tambah 1 hari
        new_time = "00" + time_part[2:]
        new_date = pd.to_datetime(date_part) + pd.Timedelta(days=1)
        return f"{new_date.strftime('%Y-%m-%d')} {new_time}"
    return ts

# perbaiki timestamp
tidal_data['timestamp'] = tidal_data['timestamp'].astype(str).apply(fix_24_hour)

# baru convert
tidal_data['timestamp'] = pd.to_datetime(tidal_data['timestamp'])

# asumsi anchor sama
anchor = pd.Timestamp("2025-01-01 00:00:00")

tidal_data['t'] = (
    (tidal_data['timestamp'] - anchor)
    .dt.total_seconds() // 3600
).astype(int)

# mapping
tidal_map = {
    (row.port_name, row.t): row.tidal_elevation
    for row in tidal_data.itertuples()
}

def get_elevation(port, t):
    return tidal_map.get((port, t), np.nan)


# =========================
# AMBIL MIN ELEVATION UNTUK YANG FAILED
# =========================
min_elevations = []

for row in result_df.itertuples():

    if row.valid:
        min_elevations.append(np.nan)
        continue

    elevations = [
        get_elevation(row.port, t)
        for t in row.failed_t
    ]

    # buang nan
    elevations = [e for e in elevations if not pd.isna(e)]

    if len(elevations) == 0:
        min_elevations.append(np.nan)
    else:
        min_elevations.append(min(elevations))

result_df['min_failed_elevation'] = min_elevations


failed = result_df[result_df['valid'] == False]

display(failed[['job_id','ship_name','port','op_seq','failed_t','min_failed_elevation']])

,job_id,ship_name,port,op_seq,failed_t,min_failed_elevation
